# Full statistics versus the actual diagonal conditioner

[Formal argument](../01_hse_fixed_dimensional_sufficiency.md). Retain the original dense-information witness, then exercise the production tokenizer. This is a finite counterexample and positive control, not trained HSE evidence.

## 1. Original full-statistic witness
The third observation row couples the two coordinates: $J$ is not diagonal. A likelihood-null perturbation preserves $b$ and the latent-dependent likelihood.

In [ ]:
import numpy as np
from hse_laplace.acquisition import gaussian_information_statistics, gaussian_posterior, GaussianInformation
from hse_laplace.conditioning import information_tokens_from_diagonal
A = np.array([[1., 0.], [0., 1.], [1., 1.]])
x1 = np.array([0.4, -0.3, 0.2])
x2 = x1 + np.array([-1., -1., 1.])
s1 = gaussian_information_statistics(A, np.eye(3), x1)
s2 = gaussian_information_statistics(A, np.eye(3), x2)
assert np.allclose(s1.score, s2.score)
assert np.allclose(s1.information, s2.information)
assert s1.information[0, 1] != 0
thetas = np.array([[-1., 0.3], [0., 0.], [0.5, -0.7], [2., 1.]])
def log_like(x, theta):
    r = x - A @ theta
    return -0.5 * r @ r
diffs = np.array([log_like(x2, t) - log_like(x1, t) for t in thetas])
assert np.ptp(diffs) < 1e-12
print('Dense-J likelihood-ratio spread:', float(np.ptp(diffs)))

## 2. Same diagonal condition, different posteriors
Here the encoder sees $(A,R,x)$ but the decoder receives only the tokens and identical coarse metadata. Construct realizable acquisitions, not just arbitrary arrays. Every exposed token field is compared.

In [ ]:
b = np.array([1., .4])
Js = [np.array([[1., .8], [.8, 1.]]), np.array([[1., -.8], [-.8, 1.]])]
operators, summaries, token_outputs, posteriors = [], [], [], []
for J in Js:
    operator = np.linalg.cholesky(J).T
    x = np.linalg.solve(operator.T, b)
    summary = gaussian_information_statistics(operator, np.eye(2), x)
    token = information_tokens_from_diagonal(summary,
        token_time_s=np.tile([0., .1], (2, 1)),
        token_band_hz=np.tile([10., 20.], (2, 1)),
        observation_reliability=np.ones(2))
    operators.append(operator)
    summaries.append(summary)
    token_outputs.append(token)
    posteriors.append(gaussian_posterior(np.zeros(2), np.eye(2), summary))
for field in token_outputs[0].__dataclass_fields__:
    np.testing.assert_allclose(getattr(token_outputs[0], field),
                               getattr(token_outputs[1], field), rtol=0, atol=1e-12)
p, q = posteriors
delta = q.mean - p.mean
kl = .5 * (np.trace(np.linalg.solve(q.covariance, p.covariance))
    + delta @ np.linalg.solve(q.covariance, delta) - 2
    + np.linalg.slogdet(q.covariance)[1] - np.linalg.slogdet(p.covariance)[1])
np.testing.assert_allclose(p.mean, [.5, 0.], atol=1e-12)
np.testing.assert_allclose(q.mean, [29/42, 10/21], atol=1e-12)
assert np.linalg.norm(delta) > .5
np.testing.assert_allclose(kl, 4/7, atol=1e-12)
print('Posterior means:', p.mean, q.mean)
print('Mean distance:', float(np.linalg.norm(delta)))
print('Directed cross-posterior KL (not compression MI):', float(kl))

## 3. Full side information removes the claimed collision
Now all compared decoders receive $a=(A,R)$. Reconstruct $J$ from this actual side input and $b$ from the token. This is a different, explicitly declared information regime—not a hidden advantage for one method.

In [ ]:
for operator, token, exact in zip(operators, token_outputs, posteriors):
    R = np.eye(operator.shape[0])
    rebuilt_J = operator.T @ np.linalg.solve(R, operator)
    rebuilt = gaussian_posterior(np.zeros(2), np.eye(2),
        GaussianInformation(token.tokens[:, 0], rebuilt_J))
    np.testing.assert_allclose(rebuilt.mean, exact.mean, atol=1e-12)
    np.testing.assert_allclose(rebuilt.covariance, exact.covariance, atol=1e-12)
assert not np.allclose(operators[0], operators[1])
print('Both full-descriptor posterior reconstructions agree with their full-statistic oracle.')

**Conclusion.** The original complete-statistic theorem survives. The actual diagonal tokens are not universally sufficient without informative side inputs. This counterexample neither proves a learned-model advantage nor that a per-mode small block is sufficient. Cross-posterior KL is not conditional MI.

In [ ]:
print("THEORY_DEMO_PASS::01_hse_fixed_dimensional_sufficiency")